### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [2]:
# ==========================================
# STEP 1: IMPORT NECESSARY MODULES & LIBARIES
# ==========================================

# Import TextLoader to read and parse local text documents
from langchain_community.document_loaders import TextLoader,DirectoryLoader

# Import character-based splitter to split text content into manageable chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import HuggingFaceEmbeddings to convert text chunks into numerical vectors
from langchain_openai import OpenAIEmbeddings

# Import FAISS vector store for semantic indexing and fast similarity searches
from langchain_community.vectorstores import FAISS

# Import the factory function to initialize a language model instance dynamically
from langchain.chat_models import init_chat_model

# Import PromptTemplate to define reusable structured templates for the LLM
from langchain_core.prompts import PromptTemplate

# Import a utility to chain an LLM with incoming retrieved document contexts
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Import a helper to combine a text retriever and a document chain into a functional RAG pipeline
from langchain_classic.chains.retrieval import create_retrieval_chain

# Import string output parser to cleanly extract text content out of LLM response structures
from langchain_core.output_parsers import StrOutputParser

# Import RunnableMap to bundle parallel execution steps in LangChain Expression Language (LCEL)
from langchain_core.runnables import RunnableMap

In [3]:
# ==========================================
# STEP 2: LOAD AND SPLIT THE DATASET
# ==========================================

# 2.1. Instantiate the loader with your specific knowledge base file
loader = TextLoader("langchain_crewai_dataset.txt")

# 2.2. Extract the text file contents into individual document structures
raw_docs = loader.load()

# 2.3. Initialize the splitter, tracking character constraints and semantic overlaps
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

# 2.4. Apply the splitting logic across the loaded document collection
chunks = splitter.split_documents(raw_docs)

In [4]:
# =====================================================================
# INSPECT GENERATED DOCUMENT CHUNKS
# =====================================================================

# Evaluate and output the split chunks array to check structural page contents and source metadata
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [ ]:
# =====================================================================
# STEP 3: INITIALIZE EMBEDDINGS AND CREATE THE RETRIEVER STORE
# =====================================================================

# Initialize the embedding transformer using SentenceTransformers all-MiniLM-L6-v2 model configuration
embedding_model=OpenAIEmbeddings()

# Pass the generated chunk array and the embedding transformer into FAISS to spin up an in-memory vector database
vectorstore=FAISS.from_documents(chunks,embedding_model)

# Configure the vector database as a LangChain retriever module
# Uses Maximal Marginal Relevance (MMR) optimization to strike a balance between similarity score and diversity in the top 5 results
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})

# Expose and confirm the configuration parameters of the retriever object
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F651D246D0>, search_type='mmr', search_kwargs={'k': 5})

In [6]:
# =====================================================================
# STEP 4: ENVIRONMENT MANAGEMENT AND CHAT MODEL CONFIGURATION
# =====================================================================

# Import standard operational tools to query runtime system environment attributes
import os

# Import tool to inspect local .env configurations and automatically apply keys to runtime execution
from dotenv import load_dotenv

# Extract matching records from file and populate local environment parameters
load_dotenv()

# Assign the API credential variable to tell downstream LangChain functions where to authentic with OpenAI
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

# Initialize the chat model utilizing the unified init_chat_model framework targeting the 'o4-mini' engine
llm=init_chat_model("openai:o4-mini")

# Return instance details to confirm configuration attributes
llm

ChatOpenAI(output_version=None, profile={'name': 'o4-mini', 'release_date': '2025-04-16', 'last_updated': '2025-04-16', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 100000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001F67D5E8CD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F67D5E9C10>, root_client=<openai.OpenAI object at 0x000001F67D5E89D0>, root_async_client=<openai.AsyncOpenAI object at 0x000001F650E01550>, model_name='o4-mini', model_kwargs={}, openai_

In [7]:
# =====================================================================
# STEP 5: PROMPT ENGINEERING & CHAIN DEFINITION FOR QUERY EXPANSION
# =====================================================================

# Query expansion
# Define strict instructions for rewriting incoming search prompts to cover alternative phrases and context
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

# Bind the dynamic template, the target language model instance, and string output parser using pipe syntax (LCEL)
query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()

# Verify the sequence pipeline steps
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOpenAI(output_version=None, profile={'name': 'o4-mini', 'release_date': '2025-04-16', 'last_updated': '2025-04-16', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 100000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00

In [8]:
# =====================================================================
# EVALUATE QUERY RETRIEVAL EXPANSION BEHAVIOR
# =====================================================================

# Invoke the expansion chain with a test payload to check context engineering output
query_expansion_chain.invoke({"query":"Langchain memory"})

'Expanded query:\n\n“LangChain memory” OR “LangChain memory management” OR “LangChain state persistence” OR “LangChain conversational memory modules” OR “LangChain session context retention” OR “LangChain memory buffers” OR “LangChain memory API” OR “LangChain buffer memory” OR “LangChain summary memory” OR “LangChain window memory” OR “LangChain vector memory” OR “LangChain knowledge memory” OR “LangChain long-term context retention” OR “LangChain token management” OR “LangChain memory strategies” OR “LangChain memory chains” OR “LangChain retriever-augmented memory” OR “LangChain vector store memory” OR “Chroma memory store” OR “FAISS memory store” OR “Pinecone memory store” OR “Redis memory backend” OR “SQLite memory backend” OR “prompt engineering memory” OR “LLM context window” OR “transformer context retention”'

In [9]:
# =====================================================================
# STEP 6: CONSTRUCT THE FINAL CONTEXTUAL RESPONSE PIPELINE
# =====================================================================

# RAG answering prompt
# Establish structural instruction template formatting the context data block alongside user query inputs
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

# Form standard document handler routine that binds documents into context variables inside the prompt layout
document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [10]:
# =====================================================================
# STEP 7: ASSEMBLE COMPLETE RETRIEVAL AUGMENTED GENERATION CHAIN
# =====================================================================

# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    # RunnableMap interceptor separates execution routines in parallel
    RunnableMap({
        # Pass the primary user query input untouched to the downstream pipeline elements
        "input": lambda x: x["input"],
        # Intercept the query input, execute it through the expansion chain, and route the output string to the MMR retriever
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    # Stream context arrays and user prompt tokens directly into the document stuffing chain executor
    | document_chain
)

In [ ]:
# ==============================================================================
# STEP 8: PIPELINE RUNTIME SIMULATION - LANGCHAIN MEMORY EVALUATION | QUESTION-1
# ==============================================================================

# Step 6: Run query
# Create specific target question dictionary object
query = {"input": "What types of memory does LangChain support?"}

# Print rewritten formulation to demonstrate query expansion transformation performance
print(query_expansion_chain.invoke({"query":query}))

# Execute full RAG pipeline workflow to retrieve relevant contexts and parse the text response
response = rag_pipeline.invoke(query)

# Render response payload string details to standard execution logs
print("✅ Answer:\n", response)

Expanded query:  
{  
  "expanded_input": "LangChain memory support: What types of memory modules, context‐persistence mechanisms, and state‐management strategies does the LangChain framework offer? Please include both in‐memory and external backends (e.g. BufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory, TokenBufferMemory), metadata‐store memory, JSON or SQL‐based persistence, knowledge‐graph memory, and vector‐store retriever memory using FAISS, Pinecone, Chroma, Weaviate, Redis, etc. Also cover differences between ephemeral in-RAM context buffers, summary-based memories, retrieval-augmented memories and long-term persistent stores for multi‐session conversational agents."  
}
✅ Answer:
 LangChain today comes with two core “conversation” memory modules:  
1. ConversationBufferMemory – keeps the raw chat history in‐context  
2. ConversationSummaryMemory – continually compresses past turns into a running summary so you stay within token limits


In [ ]:
# =====================================================================
# STEP 9: PIPELINE RUNTIME SIMULATION - CREWAI EVALUATION | QUESTION-2
# =====================================================================

# Step 6: Run query
# Define second user evaluation prompt parameter block
query = {"input": "CrewAI agents?"}

# Preview the expanded alternative queries for the CrewAI concept
print(query_expansion_chain.invoke({"query":query}))

# Invoke full RAG application pipeline stack utilizing query expansion variables
response = rag_pipeline.invoke(query)

# Log and print the extracted contextual insights out to the notebook cell interface
print("✅ Answer:\n", response)

Expanded query:

“CrewAI agents” OR “Crew AI agents” OR “AI-based crew management agents” OR “autonomous crew scheduling agents” OR “intelligent workforce coordination agents”  
AND  
(“multi-agent system” OR “agent-based modeling” OR “intelligent agents” OR “autonomous agents”)  
AND  
(“crew planning” OR “crew scheduling” OR “personnel deployment” OR “task assignment” OR “resource allocation” OR “workforce optimization” OR “predictive scheduling”)  
AND  
(“machine learning” OR “reinforcement learning” OR “optimization algorithms” OR “constraint-based scheduling” OR “genetic algorithms”)  
AND  
(aviation OR airline OR maritime OR shipping OR rail OR logistics OR manufacturing OR hospitality OR aerospace)

This expanded query covers synonyms (CrewAI agents, AI-based crew management), technical terms (multi-agent systems, reinforcement learning, optimization), and application contexts (aviation, maritime, logistics).
✅ Answer:
 CrewAI agents are autonomous, LLM-powered “crew members” 